# 01 — Bronze Ingest

**Layer:** Bronze · **Day:** 1 · **Authoritative script:** `scripts/day1_build_lakehouse.py` (function `build_bronze`)

## Objective

Ingest the public IBM HR Attrition CSV into a Parquet bronze table with provenance metadata so downstream layers always have a reproducible, queryable starting point.

## Inputs

- `data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv` — public IBM HR dataset (1,470 rows)

## Outputs

- `lakehouse/bronze/employees_raw.parquet`
  - All raw columns preserved
  - Adds `_ingest_ts` (UTC ISO timestamp)
  - Adds `_source` (lineage tag `ibm_hr_attrition_kaggle_v1`)
  - Adds `_row_hash` (12-char MD5 over the row, for change detection)

## Business value

Bronze freezes the source contract. If anyone asks *"what did the data look like on day X?"* — bronze is the answer. Lineage columns make audit and debugging cheap and stop downstream code from silently re-shaping the source.

## Reproduce

```powershell
cd D:\Agile_HR_Copilot
.\.venv\Scripts\Activate.ps1
python scripts\day1_build_lakehouse.py
```

In [ ]:
from pathlib import Path
import pandas as pd

BRONZE = Path('../lakehouse/bronze/employees_raw.parquet')
print('exists:', BRONZE.exists())

In [ ]:
if BRONZE.exists():
    df = pd.read_parquet(BRONZE)
    print(f'rows: {len(df):,}  cols: {df.shape[1]}')
    print('lineage columns:', [c for c in df.columns if c.startswith("_")])
    df.head(3)

## Interview talking points

- Bronze is intentionally **dumb** — no cleaning, no business logic. That is why downstream issues can always be traced.
- The `_row_hash` is a cheap idempotency check: re-running the script produces the same hashes, so a CI pipeline can detect upstream drift.
- In Microsoft Fabric this would be a Delta table in OneLake; the columns and contract are unchanged.